In [1]:
import sys
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Runtime setup successful.")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Pandas version: 2.2.3
NumPy version: 2.1.3
Runtime setup successful.


In [2]:
from google.colab import files

uploaded = files.upload()

Saving hrim_historical_training_baseline.csv to hrim_historical_training_baseline.csv


In [3]:
import pandas as pd
from pathlib import Path

data_path = Path("/content/hrim_historical_training_baseline.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["structural_change_proxy"].value_counts())

Dataset shape: (3266, 11)

Columns:
['repository', 'previous_snapshot', 'current_snapshot', 'package_path', 'previous_java_files', 'previous_nonempty_lines', 'previous_fan_in', 'previous_fan_out', 'previous_total_coupling', 'previous_instability', 'structural_change_proxy']

First 5 rows:


,repository,previous_snapshot,current_snapshot,package_path,previous_java_files,previous_nonempty_lines,previous_fan_in,previous_fan_out,previous_total_coupling,previous_instability,structural_change_proxy
0,R03_jabref,2020-01-08,2024-09-16,org.jabref,7,723,56,30,86,0.348837,1
1,R03_jabref,2020-01-08,2024-09-16,org.jabref.architecture,1,9,2,0,2,0.000000,1
2,R03_jabref,2020-01-08,2024-09-16,org.jabref.benchmarks,1,119,0,15,15,1.000000,0
3,R03_jabref,2020-01-08,2024-09-16,org.jabref.cli,6,887,2,28,30,0.933333,0
4,R03_jabref,2020-01-08,2024-09-16,org.jabref.gui,25,4120,47,77,124,0.620968,0



Missing values:
repository                 0
previous_snapshot          0
current_snapshot           0
package_path               0
previous_java_files        0
previous_nonempty_lines    0
previous_fan_in            0
previous_fan_out           0
previous_total_coupling    0
previous_instability       0
structural_change_proxy    0
dtype: int64

Target distribution:
structural_change_proxy
0    3030
1     236
Name: count, dtype: int64


Prepare baseline model

In [4]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

print("Scikit-learn and XGBoost imports successful.")

Scikit-learn and XGBoost imports successful.


Train and evaluate the historical baseline

In [5]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/hrim_historical_training_baseline.csv")

# Six historical architecture features
feature_cols = [
    "previous_java_files",
    "previous_nonempty_lines",
    "previous_fan_in",
    "previous_fan_out",
    "previous_total_coupling",
    "previous_instability",
]

X = df[feature_cols]
y = df["structural_change_proxy"].astype(int)
groups = df["repository"]

logo = LeaveOneGroupOut()
results = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    results.append({
        "repository": groups.iloc[test_idx].iloc[0],
        "test_examples": len(test_idx),
        "positive_examples": int(y_test.sum()),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "pr_auc": average_precision_score(y_test, probabilities),
        "roc_auc": (
            roc_auc_score(y_test, probabilities)
            if y_test.nunique() == 2 else np.nan
        ),
    })

results_df = pd.DataFrame(results)

display(results_df.round(4))

print("\nMean metrics across held-out repositories:")
display(
    results_df[
        ["precision", "recall", "f1", "pr_auc", "roc_auc"]
    ].mean().to_frame("mean").round(4)
)

,repository,test_examples,positive_examples,precision,recall,f1,pr_auc,roc_auc
0,R03_jabref,329,37,0.2407,0.3514,0.2857,0.2330,0.6982
1,R04_argouml,143,24,0.1892,0.2917,0.2295,0.1805,0.4981
2,R05_lucene,401,33,0.1132,0.3636,0.1727,0.1179,0.5396
3,R06_hadoop,1006,43,0.1212,0.3721,0.1829,0.1164,0.7404
4,R07_junit,106,16,0.2571,0.5625,0.3529,0.3362,0.6448
5,R08_apache_maven,84,7,0.1622,0.8571,0.2727,0.2771,0.6883
6,R09_kafca,327,39,0.2209,0.4872,0.3040,0.2001,0.6594
7,R11_key_cloak,870,37,0.0938,0.4865,0.1572,0.0814,0.6877



Mean metrics across held-out repositories:


,mean
precision,0.1748
recall,0.4715
f1,0.2447
pr_auc,0.1928
roc_auc,0.6446


In [6]:
from google.colab import files

uploaded = files.upload()

Saving hrim_fused_features.csv to hrim_fused_features.csv


Inspect fuse features

In [9]:
from google.colab import files

uploaded = files.upload()

Saving historical.zip to historical.zip


Extract the historical ZIP file

In [10]:
import zipfile
from pathlib import Path

zip_path = Path("/content/historical.zip")
extract_path = Path("/content/data/historical")

if not zip_path.exists():
    print("ZIP file not found. Please check the uploaded filename.")
else:
    extract_path.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    csv_files = list(extract_path.rglob("all_snapshots_architecture.csv"))

    print("Extraction completed.")
    print("Historical snapshot CSV files found:", len(csv_files))

    for file in csv_files:
        print(file)

Extraction completed.
Historical snapshot CSV files found: 11
/content/data/historical/historical/R09_kafca/all_snapshots_architecture.csv
/content/data/historical/historical/R03_jabref/all_snapshots_architecture.csv
/content/data/historical/historical/R02_common_lang/all_snapshots_architecture.csv
/content/data/historical/historical/R05_lucene/all_snapshots_architecture.csv
/content/data/historical/historical/R06_hadoop/all_snapshots_architecture.csv
/content/data/historical/historical/R11_key_cloak/all_snapshots_architecture.csv
/content/data/historical/historical/R04_argouml/all_snapshots_architecture.csv
/content/data/historical/historical/R01_spring_petclinic/all_snapshots_architecture.csv
/content/data/historical/historical/R08_apache_maven/all_snapshots_architecture.csv
/content/data/historical/historical/R10_teammates/all_snapshots_architecture.csv
/content/data/historical/historical/R07_junit/all_snapshots_architecture.csv


Inspect the historical snapshot coverage

In [11]:
from pathlib import Path
import pandas as pd

historical_root = Path("/content/data/historical/historical")

csv_files = sorted(
    historical_root.glob("*/all_snapshots_architecture.csv")
)

print("Repositories found:", len(csv_files))

for file in csv_files:
    df = pd.read_csv(file)

    print(
        f"{file.parent.name}: "
        f"{len(df)} rows, "
        f"{df['snapshot_date'].nunique()} snapshots, "
        f"{df['package_path'].nunique()} unique packages"
    )

    if file == csv_files[0]:
        print("\nColumn names:")
        print(df.columns.tolist())
        print("\nFirst five rows:")
        display(df.head())

Repositories found: 11
R01_spring_petclinic: 23 rows, 4 snapshots, 14 unique packages

Column names:
['repository', 'snapshot_date', 'commit_hash', 'package_path', 'java_files', 'nonempty_lines', 'fan_in', 'fan_out', 'total_coupling', 'instability']

First five rows:


,repository,snapshot_date,commit_hash,package_path,java_files,nonempty_lines,fan_in,fan_out,total_coupling,instability
0,R01_spring_petclinic,2013-01-09,f09d67cc1c216b445d01f1893f37ff391bb3b4b7,org.springframework.samples.petclinic,1,10,0,0,0,0.0
1,R01_spring_petclinic,2013-01-09,f09d67cc1c216b445d01f1893f37ff391bb3b4b7,org.springframework.samples.petclinic.owner,4,80,0,1,1,1.0
2,R01_spring_petclinic,2013-01-09,f09d67cc1c216b445d01f1893f37ff391bb3b4b7,org.springframework.samples.petclinic.pet,4,55,0,1,1,1.0
3,R01_spring_petclinic,2013-01-09,f09d67cc1c216b445d01f1893f37ff391bb3b4b7,org.springframework.samples.petclinic.util,3,15,2,0,2,0.0
4,R01_spring_petclinic,2015-01-21,cc4ae966c63610344313e6c7bf81ae7219ab8370,org.springframework.samples.petclinic.model,11,621,7,0,7,0.0


R02_common_lang: 51 rows, 4 snapshots, 20 unique packages
R03_jabref: 792 rows, 4 snapshots, 462 unique packages
R04_argouml: 286 rows, 4 snapshots, 143 unique packages
R05_lucene: 921 rows, 4 snapshots, 512 unique packages
R06_hadoop: 1891 rows, 4 snapshots, 885 unique packages
R07_junit: 195 rows, 4 snapshots, 89 unique packages
R08_apache_maven: 405 rows, 4 snapshots, 280 unique packages
R09_kafca: 622 rows, 4 snapshots, 294 unique packages
R10_teammates: 80 rows, 4 snapshots, 58 unique packages
R11_key_cloak: 1770 rows, 4 snapshots, 895 unique packages
